<a href="https://colab.research.google.com/github/Eden-Sibhat/2026-HYU-AUE8088-PA2/blob/main/pa2_2024200911_EdenSibhat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AUE8088 PA2 — PyTorch Colab Run-All Notebook (Level 1–5)

이 노트북은 업로드된 Level 1–5 노트북을 하나의 Colab 파일로 정리한 버전입니다.

**Run All 목표**
- Level 1 ~ 5 실험을 순서대로 재현
- Colab T4 GPU 기준으로 처음부터 끝까지 실행
- `checkpoints/`, `level5_picks.json`, `submission/level5_submission.csv` 생성
- W&B 로그인, GitHub token push, 수동 다운로드 셀 제거

기본값은 실제 실험용 epoch로 설정되어 있습니다. 빠른 코드 점검만 하고 싶을 때는 아래 설정 셀에서 `QUICK_DEBUG = True` 로 바꾸세요.


In [1]:

# ============================================================
# 0. Global configuration
# ============================================================

# W&B is disabled by default so that Colab "Runtime > Run all" does not stop for API login.
# Set USE_WANDB = True only if you want online logging and are ready to log in manually.
USE_WANDB = False

# Full experiment by default. For a quick smoke test, set QUICK_DEBUG = True.
QUICK_DEBUG = False

SEED = 42
BATCH = 64
DATA_ROOT = "../data/set_a"
SET_B_ROOT = "../data/set_b"

# Epochs. Full values follow the uploaded notebooks.
EPOCHS_LEVEL1 = 1 if QUICK_DEBUG else 30
EPOCHS_LEVEL2 = 1 if QUICK_DEBUG else 25
EPOCHS_LEVEL3 = 1 if QUICK_DEBUG else 25
EPOCHS_LEVEL5 = 1 if QUICK_DEBUG else 25

# Level 1 backbones. Keep all three for the full Level 1 experiment.
LEVEL1_BACKBONES_TO_RUN = ["vgg16", "resnet18", "resnet50"]

# Level 5 selection size. The assignment requires max 1000.
K_LEVEL5 = 1000

# Disable W&B before importing the project logger.
import os
if not USE_WANDB:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"

print("USE_WANDB:", USE_WANDB)
print("QUICK_DEBUG:", QUICK_DEBUG)
print("Epochs:", EPOCHS_LEVEL1, EPOCHS_LEVEL2, EPOCHS_LEVEL3, EPOCHS_LEVEL5)


USE_WANDB: False
QUICK_DEBUG: False
Epochs: 30 25 25 25


In [2]:
# ============================================================
# 1. Colab setup: clone repo, install requirements, import modules
# ============================================================
import os
import sys
import subprocess
from pathlib import Path

repo_name = "2026-HYU-AUE8088-PA2"
repo_url = "https://github.com/Eden-Sibhat/2026-HYU-AUE8088-PA2"
repo_dir = Path("/content") / repo_name

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

os.chdir(repo_dir)
print("Current working directory:", os.getcwd())

# Install project dependencies and gdown for dataset download.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "gdown"], check=True)

# Create output folders inside the repository so all later levels can find the files.
Path("checkpoints").mkdir(exist_ok=True)
Path("../submission").mkdir(exist_ok=True)


Current working directory: /content/2026-HYU-AUE8088-PA2


In [3]:

# ============================================================
# 2. Common imports and helper functions
# ============================================================
import os
import sys
import json
import zipfile
import random
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

# Optional imports used in later levels.
from src.models.vit import vit_small_patch16_224
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss
from src.augment.mix import mixup_data, cutmix_data, mixed_loss

set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

WANDB_PROJECT = "aue8088-pa2" if USE_WANDB else None
if USE_WANDB:
    import wandb
    wandb.login()

LOSS_WEIGHTS = {a: 1.0 for a in ATTRIBUTES}


def make_train_config(epochs, loss_weights=None):
    """Create TrainConfig safely and avoid cfg.loss_weights=None errors."""
    if loss_weights is None:
        loss_weights = {a: 1.0 for a in ATTRIBUTES}
    try:
        return TrainConfig(epochs=epochs, loss_weights=loss_weights)
    except TypeError:
        cfg = TrainConfig(epochs=epochs)
        cfg.loss_weights = loss_weights
        return cfg


def ensure_data(data_root=DATA_ROOT):
    """Download and extract the PA2 dataset only if ../data/set_a is missing."""
    gdrive_file_id = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
    zip_path = Path("../aue8088_pa2_data.zip")
    extract_to = Path("..")

    if Path(data_root).is_dir():
        print(f"Dataset already exists -> {data_root}")
        return

    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not zip_path.exists():
        print("Downloading dataset zip from Google Drive...")
        gdown.download(id=gdrive_file_id, output=str(zip_path), quiet=False)

    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_to)
    print(f"Done -> {data_root}")


def make_set_a_loaders(batch=BATCH, sampler=None, train_tf=None, val_tf=None):
    """Create Set A train/val dataloaders with fixed seed."""
    ensure_data(DATA_ROOT)
    if train_tf is None:
        train_tf = train_transform()
    if val_tf is None:
        val_tf = eval_transform()

    train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_tf)
    val_ds = BDDAttrDataset(DATA_ROOT, split="val", transform=val_tf)

    g = torch.Generator()
    g.manual_seed(SEED)

    if sampler is None:
        train_loader = DataLoader(
            train_ds,
            batch_size=batch,
            shuffle=True,
            num_workers=2,
            worker_init_fn=seed_worker,
            generator=g,
            pin_memory=True,
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=batch,
            sampler=sampler,
            num_workers=2,
            pin_memory=True,
        )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    return train_ds, val_ds, train_loader, val_loader


def cleanup_cuda(*objects):
    """Release GPU memory between levels."""
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


ensure_data(DATA_ROOT)
print("Attributes:", ATTRIBUTES)


device: cuda
GPU: Tesla T4


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=09c5d34f-8a1c-4962-a6e7-1fa49b88af51
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:01<00:00, 215MB/s]


Extracting dataset...
Done -> ../data/set_a
Attributes: ('weather', 'scene', 'timeofday')



# Level 1 — Classic CNNs: VGG-16, ResNet-18, ResNet-50

산출물:
- `checkpoints/level1_vgg16.pth`
- `checkpoints/level1_resnet18.pth`
- `checkpoints/level1_resnet50.pth`
- 각 모델의 validation Avg-Macro-F1 및 속성별 Macro-F1


In [4]:

# ============================================================
# Level 1 training
# ============================================================
train_ds, val_ds, train_loader, val_loader = make_set_a_loaders(batch=BATCH)

print("Class counts in Set A train:")
for a in ATTRIBUTES:
    print(f"{a:10s} -> {train_ds.class_counts(a).tolist()}")

LEVEL1_MODEL_FNS = {
    "vgg16": VGG16,
    "resnet18": resnet18,
    "resnet50": resnet50,
}


def train_level1_model(model_fn, name, epochs=EPOCHS_LEVEL1):
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = make_train_config(epochs=epochs, loss_weights=LOSS_WEIGHTS)

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "level": 1,
            "backbone": name,
            "epochs": epochs,
            "batch": BATCH,
            "lr": 3e-4,
            "weight_decay": 5e-4,
            "seed": SEED,
            "loss_weights": LOSS_WEIGHTS,
        },
        tags=["level1", name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)
    val_metrics = trainer.evaluate(val_loader)

    print(f"\nLevel 1 / {name} final validation metrics")
    for k, v in val_metrics.items():
        if isinstance(v, (int, float)):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    ckpt_path = Path("checkpoints") / f"level1_{name}.pth"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "model_name": name,
            "history": history,
            "val_metrics": val_metrics,
            "best_avg_mf1": val_metrics.get("avg_macro_f1", None),
            "seed": SEED,
            "epochs": epochs,
        },
        ckpt_path,
    )
    print("Saved:", ckpt_path)
    return val_metrics


level1_results = {}
for name in LEVEL1_BACKBONES_TO_RUN:
    print("\n" + "=" * 70)
    print("Training Level 1 model:", name)
    metrics = train_level1_model(LEVEL1_MODEL_FNS[name], name)
    level1_results[name] = metrics
    cleanup_cuda()

print("\nLevel 1 summary")
for name, metrics in level1_results.items():
    print(name, "Avg-Macro-F1 =", metrics.get("avg_macro_f1"))


Dataset already exists -> ../data/set_a
Class counts in Set A train:
weather    -> [3100, 800, 400, 200, 0, 500]
scene      -> [3052, 1386, 562]
timeofday  -> [2479, 2129, 392]

Training Level 1 model: vgg16


[epoch 01/30] train_loss=4.1424  val_avg_MF1=0.3576  per={'weather': 0.125, 'scene': 0.30961574862767366, 'timeofday': 0.6381547116736991}


[epoch 02/30] train_loss=2.4231  val_avg_MF1=0.3600  per={'weather': 0.17481329475879703, 'scene': 0.2669387755102041, 'timeofday': 0.6382852357730443}


[epoch 03/30] train_loss=2.3151  val_avg_MF1=0.2309  per={'weather': 0.15066334991708125, 'scene': 0.30008383918998444, 'timeofday': 0.24202240387526489}


[epoch 04/30] train_loss=2.2138  val_avg_MF1=0.3994  per={'weather': 0.1889613259668508, 'scene': 0.35568815718190133, 'timeofday': 0.6536596204012963}


...

[epoch 26/30] train_loss=1.0898  val_avg_MF1=0.6128  per={'weather': 0.5060249404092667, 'scene': 0.5384145276759211, 'timeofday': 0.7939699771481057}


[epoch 27/30] train_loss=1.0604  val_avg_MF1=0.6326  per={'weather': 0.5359778246542953, 'scene': 0.5609435392754357, 'timeofday': 0.8008353898110113}


[epoch 28/30] train_loss=1.0216  val_avg_MF1=0.6257  per={'weather': 0.533053165915637, 'scene': 0.5322207702337507, 'timeofday': 0.811852031655449}


[epoch 29/30] train_loss=1.0222  val_avg_MF1=0.6341  per={'weather': 0.5283099759843947, 'scene': 0.5620292561469032, 'timeofday': 0.811841403290428}


[epoch 30/30] train_loss=0.9852  val_avg_MF1=0.6377  per={'weather': 0.5334524826515268, 'scene': 0.5715429054087874, 'timeofday': 0.8080713536452698}

Level 1 / resnet50 final validation metrics
avg_macro_f1: 0.6377
per_macro_f1: {'weather': 0.5334524826515268, 'scene': 0.5715429054087874, 'timeofday': 0.8080713536452698}
preds: {'weather': array([0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 5, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 3, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5,
       0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,


# Level 2 — Vision Transformer: ViT-S/16

산출물:
- `checkpoints/level2_best.pth`
- ViT-S/16 validation metrics

`timm` 또는 `torchvision.models` 를 사용하지 않고, repository 내부의 직접 구현 모델을 사용합니다.


In [2]:

# ============================================================
# Level 2 training: ViT-S/16
# ============================================================
train_ds, val_ds, train_loader, val_loader = make_set_a_loaders(batch=BATCH)

USE_PRETRAINED = False
set_seed(SEED, deterministic=True)
model = vit_small_patch16_224().to(device)

epochs = EPOCHS_LEVEL2
optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
cfg = make_train_config(epochs=epochs, loss_weights=LOSS_WEIGHTS)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level2-vit_s16{'-pretrained' if USE_PRETRAINED else ''}",
    config={
        "level": 2,
        "backbone": "vit_s16",
        "pretrained": USE_PRETRAINED,
        "epochs": epochs,
        "batch": BATCH,
        "lr": 5e-4,
        "weight_decay": 5e-2,
        "seed": SEED,
        "loss_weights": LOSS_WEIGHTS,
    },
    tags=["level2", "vit_s16"],
)

trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
history = trainer.fit(train_loader, val_loader)
val_metrics = trainer.evaluate(val_loader)

print("Level 2 final validation metrics")
for k, v in val_metrics.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
logger.finish()

ckpt_path = Path("checkpoints/level2_best.pth")
torch.save(
    {
        "state_dict": model.state_dict(),
        "model_name": "vit_s16",
        "pretrained": USE_PRETRAINED,
        "history": history,
        "val_metrics": val_metrics,
        "best_avg_mf1": val_metrics.get("avg_macro_f1", None),
        "seed": SEED,
        "epochs": epochs,
    },
    ckpt_path,
)
print("Saved:", ckpt_path)
cleanup_cuda(model, trainer)


NameError: name 'make_set_a_loaders' is not defined


# Level 3 — Imbalance Handling + Advanced Augmentation

적용 기법:
- Sampling-level: weather 기준 class-balanced sampler
- Loss-level: weather = Focal Loss, scene = Class-Balanced Loss, timeofday = CE
- Augmentation-level: Mixup / CutMix

산출물:
- `checkpoints/level3_best.pth`


In [1]:

# ============================================================
# Level 3 training: imbalance + sampler + Mixup/CutMix
# ============================================================
ensure_data(DATA_ROOT)
train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds = BDDAttrDataset(DATA_ROOT, "val", transform=eval_transform())

sampler = class_balanced_sampler(train_ds, attribute="weather")
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

samples_s = train_ds.class_counts("scene")
loss_fns = {
    "weather": FocalLoss(gamma=2.0).to(device),
    "scene": ClassBalancedLoss(samples_s).to(device),
    "timeofday": nn.CrossEntropyLoss(),
}

set_seed(SEED, deterministic=True)
model = resnet18().to(device)
epochs = EPOCHS_LEVEL3
optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
cfg = make_train_config(epochs=epochs, loss_weights=LOSS_WEIGHTS)

EXPERIMENT_NAME = "focal+sampler+mixup_cutmix"
logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME}",
    config={
        "level": 3,
        "backbone": "resnet18",
        "sampler": "class_balanced(weather)",
        "loss": {"weather": "focal_g2.0", "scene": "class_balanced", "timeofday": "ce"},
        "augmentation": "mixup_or_cutmix_each_batch",
        "epochs": epochs,
        "batch": BATCH,
        "lr": 3e-4,
        "seed": SEED,
        "loss_weights": LOSS_WEIGHTS,
    },
    tags=["level3", EXPERIMENT_NAME],
)

trainer = MultiTaskTrainer(model, optim, sched, loss_fns, device, cfg, logger=logger)


def step_with_mix(images, targets):
    """Apply Mixup or CutMix with equal probability."""
    if torch.rand(1).item() < 0.5:
        x, ya, yb, lam = mixup_data(images, targets, alpha=0.2)
    else:
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam)

from tqdm import tqdm
best_avg_mf1 = -1.0
history = []
ckpt_path = Path("checkpoints/level3_best.pth")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc=f"Level 3 Epoch {epoch}/{epochs}")
    for batch in pbar:
        images = batch["image"].to(device)
        targets = {
            "weather": batch["weather"].to(device),
            "scene": batch["scene"].to(device),
            "timeofday": batch["timeofday"].to(device),
        }

        optim.zero_grad()
        loss = step_with_mix(images, targets)
        loss.backward()
        optim.step()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({"loss": running_loss / max(num_batches, 1)})

    sched.step()
    train_loss = running_loss / max(num_batches, 1)

    model.eval()
    val_metrics = trainer.evaluate(val_loader)
    avg_mf1 = val_metrics["avg_macro_f1"]

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_avg_macro_f1={avg_mf1:.4f}")

    epoch_result = {"epoch": epoch, "train_loss": train_loss, **val_metrics}
    history.append(epoch_result)

    logger.log(
        {
            "train/loss": train_loss,
            "val/avg_macro_f1": avg_mf1,
            "val/mf1_weather": val_metrics.get("mf1_weather", None),
            "val/mf1_scene": val_metrics.get("mf1_scene", None),
            "val/mf1_timeofday": val_metrics.get("mf1_timeofday", None),
            "lr": optim.param_groups[0]["lr"],
        },
        step=epoch,
    )

    if avg_mf1 > best_avg_mf1:
        best_avg_mf1 = avg_mf1
        torch.save(
            {
                "state_dict": model.state_dict(),
                "model_name": "resnet18_level3_mixup_cutmix",
                "epoch": epoch,
                "best_avg_mf1": best_avg_mf1,
                "val_metrics": val_metrics,
                "history": history,
                "seed": SEED,
                "epochs": epochs,
            },
            ckpt_path,
        )
        print(f"Saved new best Level 3 checkpoint: {best_avg_mf1:.4f}")

print("Training finished.")
print("Best Level 3 Avg-Macro-F1:", best_avg_mf1)

# Final confusion matrix and per-class PRF table
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
prf = per_class_prf(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
    logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
logger.finish()

print("Saved:", ckpt_path)
cleanup_cuda(model, trainer)


NameError: name 'ensure_data' is not defined


# Level 4 — Grad-CAM + Efficiency Trade-off

산출물:
- 속성별 normalized confusion matrix
- Grad-CAM panel
- FPS vs Avg-Macro-F1 Pareto plot

기본 분석 모델은 Level 3 best checkpoint입니다. 없으면 Level 1 ResNet-50 checkpoint로 fallback합니다.


In [ ]:

# ============================================================
# Level 4 analysis: confusion matrix, Grad-CAM, FPS Pareto
# ============================================================
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except Exception:
    sns = None

from src.utils.efficiency import measure_fps
from src.xai.gradcam import GradCAM

ensure_data(DATA_ROOT)
val_ds = BDDAttrDataset(DATA_ROOT, "val", transform=eval_transform())
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

logger = WandbLogger(project=WANDB_PROJECT, run_name="level4-analysis", tags=["level4", "analysis"])

# Choose the best available checkpoint for analysis.
if Path("checkpoints/level3_best.pth").exists():
    analysis_ckpt_path = Path("checkpoints/level3_best.pth")
    analysis_model_name = "resnet18"
    analysis_model_fn = resnet18
elif Path("checkpoints/level1_resnet50.pth").exists():
    analysis_ckpt_path = Path("checkpoints/level1_resnet50.pth")
    analysis_model_name = "resnet50"
    analysis_model_fn = resnet50
else:
    raise FileNotFoundError("No analysis checkpoint found. Run Level 1/3 first.")

print("Analysis checkpoint:", analysis_ckpt_path)
model = analysis_model_fn().to(device)
ckpt = torch.load(analysis_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# 1) Normalized confusion matrices
preds, probs, targets, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(preds, targets, normalize="true")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, a in zip(axes, ATTRIBUTES):
    if sns is not None:
        sns.heatmap(
            cms[a], annot=True, fmt=".2f", cmap="Blues", ax=ax, cbar=False,
            xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a]
        )
    else:
        ax.imshow(cms[a])
        ax.set_xticks(range(len(CLASS_NAMES[a]))); ax.set_xticklabels(CLASS_NAMES[a], rotation=45, ha="right")
        ax.set_yticks(range(len(CLASS_NAMES[a]))); ax.set_yticklabels(CLASS_NAMES[a])
    ax.set_title(f"{a}")
    ax.set_xlabel("pred")
    ax.set_ylabel("true")
fig.tight_layout()
plt.show()
logger.log_image("analysis/confusion_matrices", fig)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"analysis/cm_{a}", cms[a], CLASS_NAMES[a])

# 2) Grad-CAM panel. This is for ResNet analysis models.
target_layer = model.layer4[-1]
gc = GradCAM(model, target_layer)
batch = next(iter(val_loader))
x = batch["image"][:6].to(device)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, attr in enumerate(ATTRIBUTES):
    cam = gc(x, lambda out, a=attr: out[a].max(dim=-1).values.sum())
    for col in range(x.shape[0]):
        img = x[col].detach().cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[row, col].imshow(img)
        axes[row, col].imshow(cam[col].detach().cpu().numpy(), cmap="jet", alpha=0.45)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(attr, fontsize=14)
fig.tight_layout()
plt.show()
logger.log_image("analysis/gradcam_panel", fig)

# 3) FPS measurement and Pareto table
fps_model_fns = {
    "vgg16": VGG16,
    "resnet18": resnet18,
    "resnet50": resnet50,
    "vit_s16": vit_small_patch16_224,
}

fps_rows = []
for name, fn in fps_model_fns.items():
    try:
        m = fn().to(device).eval()
        fps = measure_fps(m, device, batch_size=1, n_warmup=20, n_iter=200)
        fps_rows.append([name, round(float(fps), 2)])
        print(f"{name:10s} FPS = {fps:.2f}")
        cleanup_cuda(m)
    except Exception as e:
        print(f"FPS skipped for {name}: {e}")

logger.log_table("analysis/fps", ["backbone", "FPS"], fps_rows)

# Load Avg-Macro-F1 from saved checkpoints.
ckpt_map = {
    "vgg16": Path("checkpoints/level1_vgg16.pth"),
    "resnet18": Path("checkpoints/level1_resnet18.pth"),
    "resnet50": Path("checkpoints/level1_resnet50.pth"),
    "vit_s16": Path("checkpoints/level2_best.pth"),
    "level3_resnet18": Path("checkpoints/level3_best.pth"),
}

avg_mf1_dict = {}
for name, path in ckpt_map.items():
    if path.exists():
        c = torch.load(path, map_location="cpu", weights_only=False)
        val_metrics = c.get("val_metrics", {})
        avg = c.get("best_avg_mf1", None)
        if avg is None and isinstance(val_metrics, dict):
            avg = val_metrics.get("avg_macro_f1", None)
        if avg is not None:
            avg_mf1_dict[name] = float(avg)

# Use Level 3 result as the ResNet-18 improved point, and Level 1 as baseline point.
pareto_rows = []
fps_dict = {name: fps for name, fps in fps_rows}
for name, fps in fps_dict.items():
    if name in avg_mf1_dict:
        pareto_rows.append({"backbone": name, "FPS": fps, "Avg-Macro-F1": avg_mf1_dict[name]})
if "level3_resnet18" in avg_mf1_dict and "resnet18" in fps_dict:
    pareto_rows.append({"backbone": "level3_resnet18", "FPS": fps_dict["resnet18"], "Avg-Macro-F1": avg_mf1_dict["level3_resnet18"]})

import pandas as pd
pareto_df = pd.DataFrame(pareto_rows)
display(pareto_df)

if len(pareto_df) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(pareto_df["FPS"], pareto_df["Avg-Macro-F1"], s=80)
    for _, row in pareto_df.iterrows():
        ax.annotate(row["backbone"], (row["FPS"], row["Avg-Macro-F1"]), textcoords="offset points", xytext=(6, 6), ha="left")
    ax.set_xlabel("FPS, batch size = 1")
    ax.set_ylabel("Validation Avg-Macro-F1")
    ax.set_title("Efficiency–Accuracy Trade-off")
    ax.grid(True, alpha=0.3)
    plt.show()
    logger.log_table("analysis/pareto_table", ["backbone", "FPS", "Avg-Macro-F1"], pareto_df.values.tolist())
    logger.log_image("analysis/pareto", fig)

logger.finish()
cleanup_cuda(model)



# Level 5 — Data Mining Challenge: The 1,000-Pick

전략:
- Level 3 best model로 Set B score 계산
- Uncertainty score: 낮은 max-softmax confidence
- Hard-example score: 예측이 정답과 다른 task 수
- Rare-class score: Set A train에서 부족한 class 우선
- Final score = `0.4 uncertainty + 0.3 hard-example + 0.3 rare-class`

산출물:
- `level5_picks.json`
- `checkpoints/level5_final.pth`
- `../submission/level5_submission.csv`


In [ ]:

# ============================================================
# Level 5 data mining and final training
# ============================================================
ensure_data(DATA_ROOT)
if not Path("checkpoints/level3_best.pth").exists():
    raise FileNotFoundError("Level 5 requires checkpoints/level3_best.pth. Run Level 3 first.")

# 1) Score all Set B images with Level 3 best model.
model = resnet18().to(device)
ckpt = torch.load("checkpoints/level3_best.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.eval()

print("Loaded Level 3 checkpoint")
print("Best Avg-Macro-F1:", ckpt.get("best_avg_mf1", "not saved"))

set_b = BDDAttrDataset(SET_B_ROOT, split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)
print(f"uncertainty shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")

# 2) Build hybrid curation score.
labels_b = {a: [] for a in ATTRIBUTES}
for s in set_b.samples:
    labels_b["weather"].append(int(s.weather))
    labels_b["scene"].append(int(s.scene))
    labels_b["timeofday"].append(int(s.timeofday))
for a in ATTRIBUTES:
    labels_b[a] = np.array(labels_b[a])

hard_score = np.zeros(len(set_b), dtype=np.float32)
for a in ATTRIBUTES:
    hard_score += (preds_b[a] != labels_b[a]).astype(np.float32)
hard_score = hard_score / len(ATTRIBUTES)

set_a_train = BDDAttrDataset(DATA_ROOT, split="train", transform=eval_transform())
class_counts = {a: Counter() for a in ATTRIBUTES}
for s in set_a_train.samples:
    class_counts["weather"][int(s.weather)] += 1
    class_counts["scene"][int(s.scene)] += 1
    class_counts["timeofday"][int(s.timeofday)] += 1

rare_score = np.zeros(len(set_b), dtype=np.float32)
for i in range(len(set_b)):
    score = 0.0
    for a in ATTRIBUTES:
        cls = labels_b[a][i]
        count = class_counts[a][cls]
        score += 1.0 / np.sqrt(count + 1)
    rare_score[i] = score / len(ATTRIBUTES)


def normalize(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

uncertainty_norm = normalize(uncertainty)
hard_score_norm = normalize(hard_score)
rare_score_norm = normalize(rare_score)

final_score = 0.4 * uncertainty_norm + 0.3 * hard_score_norm + 0.3 * rare_score_norm
K = min(K_LEVEL5, len(set_b))
order = np.argsort(-final_score)[:K]

picks = []
for i in order:
    s = set_b.samples[i]
    reason_list = []
    if uncertainty_norm[i] > 0.7:
        reason_list.append("high uncertainty")
    if hard_score_norm[i] > 0.5:
        reason_list.append("hard example")
    if rare_score_norm[i] > 0.7:
        reason_list.append("rare class")
    if len(reason_list) == 0:
        reason_list.append("high combined score")

    picks.append(
        {
            "image_id": s.image_id,
            "weather": int(s.weather),
            "scene": int(s.scene),
            "timeofday": int(s.timeofday),
            "score": float(final_score[i]),
            "uncertainty_score": float(uncertainty_norm[i]),
            "hard_score": float(hard_score_norm[i]),
            "rare_score": float(rare_score_norm[i]),
            "reason": ", ".join(reason_list),
        }
    )

with open("level5_picks.json", "w") as f:
    json.dump(
        {
            "strategy": (
                "Hybrid active data curation using uncertainty, hard-example mining, "
                "and rare-class balancing. The final score is "
                "0.4 uncertainty + 0.3 hard-example + 0.3 rare-class."
            ),
            "num_picks": len(picks),
            "weights": {"uncertainty": 0.4, "hard_example": 0.3, "rare_class": 0.3},
            "picks": picks,
        },
        f,
        indent=2,
    )

print(f"Saved {len(picks)} picks to level5_picks.json")
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    print(f"\nSelected distribution: {a}")
    for k in range(NUM_CLASSES[a]):
        print(f"  {CLASS_NAMES[a][k]:20s}: {cnt.get(k, 0)}")

cleanup_cuda(model)


In [ ]:

# ============================================================
# Level 5 final training with Set A + selected Set B picks
# ============================================================
STRATEGY_NAME = "hybrid_uncertainty_hard_rare"

extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]
train_aug = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform(), extra_picks=extra)
val_ds = BDDAttrDataset(DATA_ROOT, "val", transform=eval_transform())

g = torch.Generator()
g.manual_seed(SEED)
loader_tr = DataLoader(
    train_aug,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)
loader_val = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

set_seed(SEED, deterministic=True)
model2 = resnet18().to(device)
epochs = EPOCHS_LEVEL5
optim = torch.optim.AdamW(model2.parameters(), lr=3e-4, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
cfg = make_train_config(epochs=epochs, loss_weights=LOSS_WEIGHTS)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level5-{STRATEGY_NAME}",
    config={
        "level": 5,
        "backbone": "resnet18",
        "strategy": STRATEGY_NAME,
        "num_picks": len(picks),
        "epochs": epochs,
        "batch": BATCH,
        "lr": 3e-4,
        "seed": SEED,
        "loss_weights": LOSS_WEIGHTS,
    },
    tags=["level5", STRATEGY_NAME],
)

trainer = MultiTaskTrainer(model2, optim, sched, losses, device, cfg, logger=logger)
history = trainer.fit(loader_tr, loader_val)
val_metrics = trainer.evaluate(loader_val)

print("Level 5 final validation metrics")
for k, v in val_metrics.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

val_pred, _, val_tgt, _ = collect_predictions(model2, loader_val, device)
cms = confusion_matrices(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    cnt = Counter(p[a] for p in picks)
    rows = [[CLASS_NAMES[a][k], cnt.get(k, 0)] for k in range(NUM_CLASSES[a])]
    logger.log_table(f"picks/distribution_{a}", ["class", "count"], rows)

ckpt_path = Path("checkpoints/level5_final.pth")
torch.save(
    {
        "state_dict": model2.state_dict(),
        "history": history,
        "val_metrics": val_metrics,
        "best_avg_mf1": val_metrics.get("avg_macro_f1", None),
        "strategy": STRATEGY_NAME,
        "num_picks": len(picks),
        "seed": SEED,
        "epochs": epochs,
    },
    ckpt_path,
)
print("Saved:", ckpt_path)
logger.finish()


In [ ]:
# ============================================================
# Level 5 Kaggle submission json
# ============================================================
Path("../submission").mkdir(exist_ok=True)
model2.eval()

test_ds = BDDAttrDataset(DATA_ROOT, "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
submission_path = "../submission/level5_picks.json"
write_submission(submission_path, ids_te, preds_te)

print("Created submission:", submission_path)
for path in sorted(Path("../submission").glob("*")):
    print(f"{path}  {path.stat().st_size / 1024:.1f} KB")
cleanup_cuda(model2, trainer)



# Final output checklist

After **Runtime > Run all**, the important files should be:

```text
/content/2026-HYU-AUE8088-PA2/checkpoints/level1_vgg16.pth
/content/2026-HYU-AUE8088-PA2/checkpoints/level1_resnet18.pth
/content/2026-HYU-AUE8088-PA2/checkpoints/level1_resnet50.pth
/content/2026-HYU-AUE8088-PA2/checkpoints/level2_best.pth
/content/2026-HYU-AUE8088-PA2/checkpoints/level3_best.pth
/content/2026-HYU-AUE8088-PA2/checkpoints/level5_final.pth
/content/2026-HYU-AUE8088-PA2/level5_picks.json
/content/submission/level5_picks.json
```

GitHub push and token cells were intentionally removed because they stop automatic execution and are not required for the reproducible Colab experiment file.
